# 03. 계약서 → 엔진 팩토리 → NPV

계약서(무엇)와 평가 방법(어떻게)을 나눈다(원칙 6). 계약서는 순수 데이터이고, 엔진은 팩토리의 빌더에서 받는다.

| 단계 | QuantTool 코드 | ORE·QuantLib 원본 |
|---|---|---|
| 계약서 | `positions/ust_book.yaml`, `src/quanttool/data/portfolio/bond.py` `Bond` | `OREData/ored/portfolio/trade.hpp`, `bond.hpp`, `portfolio.xml` |
| 엔진 선택 | `configs/pricing/engines.yaml` → `EngineData` | `pricingengine.xml`, `enginedata.hpp` |
| 팩토리·빌더 | `src/quanttool/data/portfolio/enginefactory.py`, `builders/bond.py` | `enginefactory.hpp`, `builders/bond.hpp` |
| 상품↔엔진 대화 | — | `docs/architecture.md` 0.1 |

> 데이터: `tests/fixtures/ust_par_sample.csv`는 재무부 CSV와 **형식만 같은 합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE_CSV = ROOT / "tests" / "fixtures" / "ust_par_sample.csv"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.data.configuration import load_config
from quanttool.data.marketdata import TodaysMarket, UstParCsvLoader
from quanttool.data.portfolio import EngineFactory, Portfolio

cfg = load_config(ROOT / "configs")
loader = UstParCsvLoader(SAMPLE_CSV)
market = TodaysMarket.from_config(date(2026, 10, 23), cfg, loader)

## 1. 계약서는 데이터다
엔진도, 커브 객체도 모른다. 할인 커브는 **이름**(`discount: UST`)으로만 가리킨다.

In [ ]:
book = Portfolio.load(ROOT / "positions" / "ust_book.yaml")
pd.DataFrame([t.model_dump() for t in book]).set_index("id")

## 2. 엔진 팩토리와 빌더
`engines.yaml`의 `Bond: {model: DiscountedCashflows, engine: DiscountingBondEngine}`에 따라 팩토리가 빌더를 돌려준다. 빌더는 키(할인 커브 이름)마다 엔진을 한 번만 만든다.

In [ ]:
factory = EngineFactory.from_config(
    cfg, market
)  # = EngineFactory(cfg.engine_data, market, cfg.conventions)
builder = factory.builder("Bond")
print(type(builder).__name__, builder.model, builder.engine_name)
e1, e2 = builder.engine("UST"), builder.engine("UST")
print("같은 키 → 같은 엔진:", e1 is e2)

## 3. 조립: `trade.build(factory)`
`Bond.build`가 일정·관례로 `ql.FixedRateBond`를 만들고, `factory.builder('Bond').engine(self.discount)`로 엔진을 받는다 (ORE `Swap::build`도 `builder("Swap")` → `engine(ccy)` 순서).

In [ ]:
with market.active():
    book.build(factory)
    bond10 = book.trades["UST_10Y"]
    ql_bond = bond10.instrument.instrument  # 안쪽의 QuantLib 상품
    display(
        pd.DataFrame(
            [(cf.date().ISO(), cf.amount()) for cf in ql_bond.cashflows()],
            columns=["date", "amount"],
        ).tail()
    )
    display(book.npvs().round(2))
    print("합계", round(book.npv(), 2))

## 4. QuantLib으로 직접 계산한 값과 같은가

In [ ]:
with market.active():
    sch = ql.Schedule(
        ql.Date(15, 8, 2026),
        ql.Date(15, 8, 2036),
        ql.Period(ql.Semiannual),
        ql.UnitedStates(ql.UnitedStates.GovernmentBond),
        ql.Unadjusted,
        ql.Unadjusted,
        ql.DateGeneration.Backward,
        False,
    )
    direct = ql.FixedRateBond(
        1,
        100.0,
        sch,
        [0.04125],
        ql.ActualActual(ql.ActualActual.Bond),
        ql.Unadjusted,
        100.0,
        ql.Date(15, 8, 2026),
    )
    direct.setPricingEngine(ql.DiscountingBondEngine(market.yield_curve("UST")))
    print(direct.NPV() * 1e7 / 100, bond10.npv())

## 5. 같은 계약서, 다른 날짜의 시장
계약서는 그대로 두고 다른 시장으로 다시 조립한다. 평가는 각 시장의 `active()` 안에서.

In [ ]:
prev_market = TodaysMarket.from_config(date(2026, 10, 22), cfg, loader)
with prev_market.active():
    book.build(EngineFactory.from_config(cfg, prev_market))
    prev = book.npvs()
with market.active():
    book.build(EngineFactory.from_config(cfg, market))
    now = book.npvs()
pd.DataFrame({"10/22": prev, "10/23": now, "손익": now - prev}).round(2)

다음: `04_simmarket_sensitivity.ipynb` — 시뮬레이션 시장을 흔들어 민감도를 본다.